In [0]:
%pip install lightgbm -q
dbutils.library.restartPython()

# 10 - Model Evaluation & Monitoring

## 📊 Évaluation Complète des Modèles de Prévision

Ce notebook évalue de manière exhaustive les deux modèles de prévision de demande électrique (24h et 7j) et stocke les résultats dans des tables structurées pour alimenter un dashboard de monitoring.

### Tables de monitoring créées :
1. **`model_performance_global`** - Métriques globales par modèle et date d'évaluation
2. **`model_performance_by_zone`** - Performance détaillée par zone géographique
3. **`model_performance_by_time`** - Métriques par heure du jour, jour de la semaine, type de jour
4. **`model_performance_by_horizon`** - Dégradation de la précision avec l'horizon (modèle 7j)
5. **`model_error_distribution`** - Distribution des erreurs et quantiles
6. **`model_predictions_sample`** - Échantillon de prédictions pour visualisations
7. **`model_feature_importance`** - Importance des features par modèle

### Métriques calculées :
- **Précision** : MAE, RMSE, MAPE, R²
- **Biais** : erreur moyenne (sur/sous-estimation)
- **Couverture** : % prédictions dans différentes marges d'erreur
- **Pics** : précision spécifique aux heures de forte demande
- **Patterns** : performance par contexte temporel (heure, jour, saison)

In [0]:

import os
import yaml
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import mlflow
from mlflow.tracking import MlflowClient

# Chemin du projet : surchargeable via la variable d'environnement
# ENERGY_FORECAST_PROJECT_ROOT (ex: Databricks Repos/Asset Bundles), avec
# repli sur le chemin Workspace actuel pour ne rien casser en l'état.
PROJECT_ROOT = os.environ.get(
    "ENERGY_FORECAST_PROJECT_ROOT",
    "/Workspace/Users/n.jouglet23@gmail.com/energy_forecast",
)

with open(f'{PROJECT_ROOT}/config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

CATALOG = config['catalog']['name']
SCHEMA = config['catalog']['schema']

# Tables sources
ACTUAL_TABLE = f"{CATALOG}.{SCHEMA}.{config['catalog']['tables']['bronze']['load_zonal']}"
FEATURES_TABLE_24H = f"{CATALOG}.{SCHEMA}.{config['catalog']['tables']['gold']['ml_features_24h']}"
FEATURES_TABLE_7J = f"{CATALOG}.{SCHEMA}.{config['catalog']['tables']['gold']['ml_features_7j']}"

# Tables de monitoring (destination)
MONITORING_TABLES = {
    'global': f"{CATALOG}.{SCHEMA}.model_performance_global",
    'by_zone': f"{CATALOG}.{SCHEMA}.model_performance_by_zone",
    'by_time': f"{CATALOG}.{SCHEMA}.model_performance_by_time",
    'by_horizon': f"{CATALOG}.{SCHEMA}.model_performance_by_horizon",
    'error_dist': f"{CATALOG}.{SCHEMA}.model_error_distribution",
    'predictions_sample': f"{CATALOG}.{SCHEMA}.model_predictions_sample",
    'feature_importance': f"{CATALOG}.{SCHEMA}.model_feature_importance"
}

print("📊 Évaluation complète des modèles")
print(f"\n📂 Tables sources :")
print(f"   Réel (bronze)  : {ACTUAL_TABLE}")
print(f"   Features 24h   : {FEATURES_TABLE_24H}")
print(f"   Features 7j    : {FEATURES_TABLE_7J}")
print(f"\n💾 Tables de monitoring (7 tables) :")
for name, table in MONITORING_TABLES.items():
    print(f"   {name:<20}: {table}")


In [0]:
# =============================================================================
# 1. CHARGEMENT DES MODÈLES MLflow
# =============================================================================
print("🔄 Chargement des modèles MLflow...\n")

# Récupérer les noms des expériences
exp_24h = config['models']['horizon_24h']['mlflow']['experiment_name']
exp_7d = config['models']['horizon_7j']['mlflow']['experiment_name']

print(f"🎯 Expériences:")
print(f"   24h: {exp_24h}")
print(f"   7j:  {exp_7d}")

# Trouver le dernier run réussi pour chaque expérience
runs_24h = mlflow.search_runs(
    experiment_names=[exp_24h],
    filter_string="status = 'FINISHED'",
    order_by=["start_time DESC"],
    max_results=1
)

runs_7d = mlflow.search_runs(
    experiment_names=[exp_7d],
    filter_string="status = 'FINISHED'",
    order_by=["start_time DESC"],
    max_results=1
)

if runs_24h.empty or runs_7d.empty:
    dbutils.notebook.exit("⚠️ Aucun modèle entraîné trouvé. Exécutez d'abord les notebooks d'entraînement.")

# Extraire les run IDs et charger les modèles
run_id_24h = runs_24h.iloc[0]['run_id']
run_id_7d = runs_7d.iloc[0]['run_id']

print(f"\n🔵 Modèle 24h:")
print(f"   Run ID: {run_id_24h}")
print(f"   Date: {runs_24h.iloc[0]['start_time']}")
if 'metrics.test_mape' in runs_24h.columns:
    print(f"   MAPE entraînement: {runs_24h.iloc[0]['metrics.test_mape']:.2f}%")

print(f"\n🟢 Modèle 7j:")
print(f"   Run ID: {run_id_7d}")
print(f"   Date: {runs_7d.iloc[0]['start_time']}")
if 'metrics.test_mape' in runs_7d.columns:
    print(f"   MAPE entraînement: {runs_7d.iloc[0]['metrics.test_mape']:.2f}%")

# Charger les modèles
model_24h = mlflow.pyfunc.load_model(f"runs:/{run_id_24h}/model")
model_7d = mlflow.pyfunc.load_model(f"runs:/{run_id_7d}/model")

print("\n✅ Modèles chargés avec succès")

In [0]:
# =============================================================================
# 2. CHARGEMENT DES DONNÉES DE TEST (30 derniers jours)
# =============================================================================
print("\n📁 Chargement des données de test (30 derniers jours)...\n")

test_data_24h = spark.sql(f"""
    SELECT *
    FROM {FEATURES_TABLE_24H}
    WHERE target_datetime >= date_sub(current_date(), 30)
""").toPandas()

test_data_7j = spark.sql(f"""
    SELECT *
    FROM {FEATURES_TABLE_7J}
    WHERE target_datetime >= date_sub(current_date(), 30)
""").toPandas()

if test_data_24h.empty or test_data_7j.empty:
    dbutils.notebook.exit("⚠️ Aucune donnée de test disponible")

print(f"✅ Modèle 24h : {len(test_data_24h):,} lignes "
      f"({test_data_24h['target_datetime'].min()} à {test_data_24h['target_datetime'].max()})")
print(f"✅ Modèle 7j  : {len(test_data_7j):,} lignes, horizons "
      f"H+{test_data_7j['forecast_horizon_hours'].min()} à "
      f"H+{test_data_7j['forecast_horizon_hours'].max()}")
print(f"   Zones     : {sorted(test_data_24h['zone'].unique())}")

In [0]:
# =============================================================================
# 3. GÉNÉRATION DES PRÉDICTIONS
# =============================================================================
print("\n🔮 Prédictions en cours...\n")

# Obtenir la liste des features que chaque modèle attend
model_24h_info = mlflow.models.get_model_info(f"runs:/{run_id_24h}/model")
model_7d_info = mlflow.models.get_model_info(f"runs:/{run_id_7d}/model")

expected_features_24h = [input.name for input in model_24h_info.signature.inputs.inputs]
expected_features_7d = [input.name for input in model_7d_info.signature.inputs.inputs]

print(f"   Modèle 24h attend {len(expected_features_24h)} features")
print(f"   Modèle 7j attend {len(expected_features_7d)} features")

# Préparer les données de test avec seulement les features attendues, chacune
# depuis sa propre table Gold.
X_test_24h = test_data_24h[expected_features_24h].copy()

# Pour le modèle 7j: créer les features manquantes si nécessaire
# (le modèle peut avoir été entraîné avec des features qui ne sont pas dans la table Gold)
for feature in expected_features_7d:
    if feature not in test_data_7j.columns:
        if feature == 'forecast_horizon_sqrt':
            test_data_7j[feature] = np.sqrt(test_data_7j['forecast_horizon_hours'])
        elif feature == 'forecast_horizon_log1p':
            test_data_7j[feature] = np.log1p(test_data_7j['forecast_horizon_hours'])
        else:
            print(f"   ⚠️  Feature manquante non reconnue: {feature} - remplie avec 0")
            test_data_7j[feature] = 0

X_test_7d = test_data_7j[expected_features_7d].copy()

# Workaround MLflow + LightGBM categorical issues:
# Access the underlying LightGBM model directly to bypass MLflow schema enforcement
print("\n⚠️  Utilisation du modèle LightGBM sous-jacent pour contourner les problèmes de catégoriques...")

# Get underlying LightGBM models
lgb_model_24h = mlflow.sklearn.load_model(f"runs:/{run_id_24h}/model")
lgb_model_7d = mlflow.sklearn.load_model(f"runs:/{run_id_7d}/model")

# LightGBM requires zone as categorical
if 'zone' in X_test_24h.columns:
    X_test_24h['zone'] = X_test_24h['zone'].astype('category')
if 'zone' in X_test_7d.columns:
    X_test_7d['zone'] = X_test_7d['zone'].astype('category')

# Prédictions modèle 24h
print("\n🔵 Modèle 24h...")
pred_24h = lgb_model_24h.predict(X_test_24h)
test_data_24h['pred_24h'] = pred_24h

# Prédictions modèle 7j
print("🟢 Modèle 7j...")
pred_7d = lgb_model_7d.predict(X_test_7d)
test_data_7j['pred_7d'] = pred_7d

# La cible est `target_demand_mw` (demande à target_datetime), pas `demand_mw`
# (demande au moment de l'émission `issue_datetime`, utilisée comme feature).
print("\n✅ Prédictions générées")
print(f"   Modèle 24h: min={pred_24h.min():.1f}, max={pred_24h.max():.1f}, mean={pred_24h.mean():.1f} MW")
print(f"   Modèle 7j:  min={pred_7d.min():.1f}, max={pred_7d.max():.1f}, mean={pred_7d.mean():.1f} MW")
print(f"   Réel 24h:  min={test_data_24h['target_demand_mw'].min():.1f}, max={test_data_24h['target_demand_mw'].max():.1f}, mean={test_data_24h['target_demand_mw'].mean():.1f} MW")
print(f"   Réel 7j:   min={test_data_7j['target_demand_mw'].min():.1f}, max={test_data_7j['target_demand_mw'].max():.1f}, mean={test_data_7j['target_demand_mw'].mean():.1f} MW")

In [0]:
# =============================================================================
# 3b. ENRICHISSEMENT DES FEATURES TEMPORELLES POUR L'ANALYSE
# =============================================================================
print("\n🕐 Ajout des features temporelles pour l'analyse...\n")

# Extraire les features temporelles de target_datetime
test_data_24h['target_hour'] = pd.to_datetime(test_data_24h['target_datetime']).dt.hour
test_data_24h['target_dayofweek'] = pd.to_datetime(test_data_24h['target_datetime']).dt.dayofweek
test_data_24h['target_is_weekend'] = (test_data_24h['target_dayofweek'] >= 5).astype(int)

test_data_7j['target_hour'] = pd.to_datetime(test_data_7j['target_datetime']).dt.hour
test_data_7j['target_dayofweek'] = pd.to_datetime(test_data_7j['target_datetime']).dt.dayofweek  
test_data_7j['target_is_weekend'] = (test_data_7j['target_dayofweek'] >= 5).astype(int)

print("✅ Features temporelles ajoutées:")
print(f"   - target_hour (0-23)")
print(f"   - target_dayofweek (0=Lundi, 6=Dimanche)")
print(f"   - target_is_weekend (0=Weekday, 1=Weekend)")

## 4. Fonctions d'aide pour le calcul des métriques

In [0]:
# =============================================================================
# 4. FONCTIONS D'AIDE POUR LE CALCUL DES MÉTRIQUES
# =============================================================================
from sklearn.metrics import r2_score
from scipy import stats

def calculate_comprehensive_metrics(y_true, y_pred, model_name=""):
    """
    Calcule un ensemble complet de métriques de performance.
    
    Returns:
        dict: Dictionnaire contenant toutes les métriques
    """
    # Métriques de base
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2 = r2_score(y_true, y_pred)
    
    # Biais (tendance à sur/sous-estimer)
    bias = (y_pred - y_true).mean()
    bias_pct = (bias / y_true.mean()) * 100
    
    # Métriques de couverture (% dans différentes marges d'erreur)
    errors_pct = np.abs((y_pred - y_true) / y_true) * 100
    coverage_1pct = (errors_pct <= 1).mean() * 100
    coverage_2pct = (errors_pct <= 2).mean() * 100
    coverage_5pct = (errors_pct <= 5).mean() * 100
    coverage_10pct = (errors_pct <= 10).mean() * 100
    
    # Quantiles des erreurs
    errors_abs = np.abs(y_pred - y_true)
    q25, q50, q75, q90, q95, q99 = np.percentile(errors_abs, [25, 50, 75, 90, 95, 99])
    
    # Statistiques descriptives
    n_samples = len(y_true)
    mean_actual = y_true.mean()
    std_actual = y_true.std()
    mean_pred = y_pred.mean()
    std_pred = y_pred.std()
    
    return {
        # Métriques principales
        'mae': float(mae),
        'rmse': float(rmse),
        'mape': float(mape),
        'r2': float(r2),
        
        # Biais
        'bias_mw': float(bias),
        'bias_pct': float(bias_pct),
        
        # Couverture
        'coverage_1pct': float(coverage_1pct),
        'coverage_2pct': float(coverage_2pct),
        'coverage_5pct': float(coverage_5pct),
        'coverage_10pct': float(coverage_10pct),
        
        # Quantiles d'erreur
        'error_q25': float(q25),
        'error_q50': float(q50),
        'error_q75': float(q75),
        'error_q90': float(q90),
        'error_q95': float(q95),
        'error_q99': float(q99),
        
        # Stats descriptives
        'n_samples': int(n_samples),
        'mean_actual': float(mean_actual),
        'std_actual': float(std_actual),
        'mean_pred': float(mean_pred),
        'std_pred': float(std_pred)
    }

def print_metrics(metrics, title=""):
    """
    Affiche les métriques de manière formatée.
    """
    if title:
        print(f"\n{'='*80}")
        print(f"{title}")
        print(f"{'='*80}")
    
    print(f"\n🎯 Métriques principales:")
    print(f"   MAE:  {metrics['mae']:>10,.1f} MW")
    print(f"   RMSE: {metrics['rmse']:>10,.1f} MW")
    print(f"   MAPE: {metrics['mape']:>10.2f} %")
    print(f"   R²:   {metrics['r2']:>10.3f}")
    
    print(f"\n🎯 Biais (sur/sous-estimation):")
    print(f"   Biais absolu: {metrics['bias_mw']:>10,.1f} MW")
    print(f"   Biais relatif: {metrics['bias_pct']:>9.2f} %")
    if abs(metrics['bias_pct']) < 0.5:
        print(f"   ✅ Biais négligeable (< 0.5%)")
    elif abs(metrics['bias_pct']) < 1.0:
        print(f"   ⚠️  Biais léger (< 1%)")
    else:
        print(f"   ❌ Biais significatif (≥ 1%)")
    
    print(f"\n🎯 Couverture (% prédictions dans la marge):")
    print(f"   Dans ±1%:  {metrics['coverage_1pct']:>6.1f}%")
    print(f"   Dans ±2%:  {metrics['coverage_2pct']:>6.1f}%")
    print(f"   Dans ±5%:  {metrics['coverage_5pct']:>6.1f}%")
    print(f"   Dans ±10%: {metrics['coverage_10pct']:>6.1f}%")
    
    print(f"\n🎯 Quantiles d'erreur absolue:")
    print(f"   Q25:  {metrics['error_q25']:>8,.1f} MW")
    print(f"   Q50:  {metrics['error_q50']:>8,.1f} MW (médiane)")
    print(f"   Q75:  {metrics['error_q75']:>8,.1f} MW")
    print(f"   Q90:  {metrics['error_q90']:>8,.1f} MW")
    print(f"   Q95:  {metrics['error_q95']:>8,.1f} MW")
    print(f"   Q99:  {metrics['error_q99']:>8,.1f} MW")
    
    print(f"\n📊 Statistiques descriptives:")
    print(f"   N échantillons: {metrics['n_samples']:,}")
    print(f"   Demande réelle:  {metrics['mean_actual']:>8,.1f} ± {metrics['std_actual']:,.1f} MW")
    print(f"   Prédiction:      {metrics['mean_pred']:>8,.1f} ± {metrics['std_pred']:,.1f} MW")

print("✅ Fonctions d'aide chargées")

## 5. Métriques Globales

In [0]:
# =============================================================================
# 5. MÉTRIQUES GLOBALES
# =============================================================================
print("\n🎯 CALCUL DES MÉTRIQUES GLOBALES\n")

# Modèle 24h - filtrer les NaN avant calcul des métriques
mask_24h = ~(test_data_24h['target_demand_mw'].isna() | test_data_24h['pred_24h'].isna())
metrics_24h_global = calculate_comprehensive_metrics(
    test_data_24h.loc[mask_24h, 'target_demand_mw'],
    test_data_24h.loc[mask_24h, 'pred_24h'],
    model_name="24h"
)
if (~mask_24h).sum() > 0:
    print(f"⚠️  {(~mask_24h).sum()} lignes avec NaN exclues du calcul (modèle 24h)\n")
print_metrics(metrics_24h_global, "🔵 MODÈLE 24H - PERFORMANCE GLOBALE")

# Modèle 7j (tous horizons confondus) - filtrer les NaN avant calcul des métriques
mask_7d = ~(test_data_7j['target_demand_mw'].isna() | test_data_7j['pred_7d'].isna())
metrics_7d_global = calculate_comprehensive_metrics(
    test_data_7j.loc[mask_7d, 'target_demand_mw'],
    test_data_7j.loc[mask_7d, 'pred_7d'],
    model_name="7j"
)
if (~mask_7d).sum() > 0:
    print(f"⚠️  {(~mask_7d).sum()} lignes avec NaN exclues du calcul (modèle 7j)\n")
print_metrics(metrics_7d_global, "\n🟢 MODÈLE 7J - PERFORMANCE GLOBALE (tous horizons)")

# Comparaison
print(f"\n\n{'='*80}")
print("🔄 COMPARAISON DES MODÈLES")
print(f"{'='*80}")

comparison_df = pd.DataFrame({
    'Modèle': ['24h', '7j (tous horizons)'],
    'MAPE (%)': [metrics_24h_global['mape'], metrics_7d_global['mape']],
    'MAE (MW)': [metrics_24h_global['mae'], metrics_7d_global['mae']],
    'RMSE (MW)': [metrics_24h_global['rmse'], metrics_7d_global['rmse']],
    'R²': [metrics_24h_global['r2'], metrics_7d_global['r2']],
    'Biais (%)': [metrics_24h_global['bias_pct'], metrics_7d_global['bias_pct']],
    'Couv. ±2%': [f"{metrics_24h_global['coverage_2pct']:.1f}%", f"{metrics_7d_global['coverage_2pct']:.1f}%"],
    'Couv. ±5%': [f"{metrics_24h_global['coverage_5pct']:.1f}%", f"{metrics_7d_global['coverage_5pct']:.1f}%"],
    'N échantillons': [metrics_24h_global['n_samples'], metrics_7d_global['n_samples']]
})

print("\n")
display(comparison_df)

# Verdict
print("\n🎯 ÉVALUATION PAR RAPPORT AUX OBJECTIFS:")
if metrics_24h_global['mape'] < 2.0:
    print(f"   ✅ Modèle 24h: EXCELLENT ({metrics_24h_global['mape']:.2f}% < 2.0%)")
elif metrics_24h_global['mape'] < 2.5:
    print(f"   ⚠️  Modèle 24h: ACCEPTABLE ({metrics_24h_global['mape']:.2f}% < 2.5%)")
else:
    print(f"   ❌ Modèle 24h: INSUFFISANT ({metrics_24h_global['mape']:.2f}% ≥ 2.5%)")

if metrics_7d_global['mape'] < 3.5:
    print(f"   ✅ Modèle 7j: EXCELLENT ({metrics_7d_global['mape']:.2f}% < 3.5%)")
elif metrics_7d_global['mape'] < 4.0:
    print(f"   ⚠️  Modèle 7j: ACCEPTABLE ({metrics_7d_global['mape']:.2f}% < 4.0%)")
else:
    print(f"   ❌ Modèle 7j: INSUFFISANT ({metrics_7d_global['mape']:.2f}% ≥ 4.0%)")

%md
## 6. Métriques par Zone Géographique

In [0]:
# =============================================================================
# 6. MÉTRIQUES PAR ZONE GÉOGRAPHIQUE
# =============================================================================
print("\n\n🗺️ CALCUL DES MÉTRIQUES PAR ZONE\n")

# Modèle 24h par zone
zone_metrics_24h = []
for zone in sorted(test_data_24h['zone'].unique()):
    zone_data = test_data_24h[test_data_24h['zone'] == zone]
    # Filtrer les NaN avant calcul des métriques
    mask = ~(zone_data['target_demand_mw'].isna() | zone_data['pred_24h'].isna())
    metrics = calculate_comprehensive_metrics(
        zone_data.loc[mask, 'target_demand_mw'],
        zone_data.loc[mask, 'pred_24h']
    )
    metrics['zone'] = zone
    metrics['model_type'] = '24h'
    zone_metrics_24h.append(metrics)

zone_df_24h = pd.DataFrame(zone_metrics_24h)

# Modèle 7j par zone
zone_metrics_7d = []
for zone in sorted(test_data_7j['zone'].unique()):
    zone_data = test_data_7j[test_data_7j['zone'] == zone]
    # Filtrer les NaN avant calcul des métriques
    mask = ~(zone_data['target_demand_mw'].isna() | zone_data['pred_7d'].isna())
    metrics = calculate_comprehensive_metrics(
        zone_data.loc[mask, 'target_demand_mw'],
        zone_data.loc[mask, 'pred_7d']
    )
    metrics['zone'] = zone
    metrics['model_type'] = '7j'
    zone_metrics_7d.append(metrics)

zone_df_7d = pd.DataFrame(zone_metrics_7d)

# Affichage des résultats
print("🔵 MODÈLE 24H - Top 5 zones les plus difficiles à prédire (MAPE):")
display(zone_df_24h.nlargest(5, 'mape')[['zone', 'mape', 'mae', 'rmse', 'r2', 'n_samples']])

print("\n🔵 MODÈLE 24H - Top 5 zones les mieux prédites (MAPE):")
display(zone_df_24h.nsmallest(5, 'mape')[['zone', 'mape', 'mae', 'rmse', 'r2', 'n_samples']])

print("\n🟢 MODÈLE 7J - Top 5 zones les plus difficiles à prédire (MAPE):")
display(zone_df_7d.nlargest(5, 'mape')[['zone', 'mape', 'mae', 'rmse', 'r2', 'n_samples']])

print("\n🟢 MODÈLE 7J - Top 5 zones les mieux prédites (MAPE):")
display(zone_df_7d.nsmallest(5, 'mape')[['zone', 'mape', 'mae', 'rmse', 'r2', 'n_samples']])

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Graphique 1: MAPE par zone (modèle 24h)
ax1 = axes[0]
zone_df_24h_sorted = zone_df_24h.sort_values('mape', ascending=False)
ax1.barh(zone_df_24h_sorted['zone'], zone_df_24h_sorted['mape'], color='blue', alpha=0.7)
ax1.axvline(x=2.0, color='red', linestyle='--', linewidth=2, label='Cible 24h (2%)')
ax1.set_xlabel('MAPE (%)', fontsize=12)
ax1.set_title('🔵 Modèle 24h - MAPE par zone', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='x')

# Graphique 2: MAPE par zone (modèle 7j)
ax2 = axes[1]
zone_df_7d_sorted = zone_df_7d.sort_values('mape', ascending=False)
ax2.barh(zone_df_7d_sorted['zone'], zone_df_7d_sorted['mape'], color='green', alpha=0.7)
ax2.axvline(x=3.5, color='orange', linestyle='--', linewidth=2, label='Cible 7j (3.5%)')
ax2.set_xlabel('MAPE (%)', fontsize=12)
ax2.set_title('🟢 Modèle 7j - MAPE par zone', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"\n✅ Métriques par zone calculées: {len(zone_df_24h)} zones pour le modèle 24h, {len(zone_df_7d)} zones pour le modèle 7j")

## 7. Métriques par Patterns Temporels

In [0]:
# =============================================================================
# 7. MÉTRIQUES PAR PATTERNS TEMPORELS
# =============================================================================
print("\n\n🕒 CALCUL DES MÉTRIQUES PAR PATTERNS TEMPORELS\n")

# A. Métriques par heure du jour
print("🕒 Métriques par heure du jour (0-23h)...")

# Modèle 24h - filtrer les NaN avant le groupby
mask_24h_temporal = ~(test_data_24h['target_demand_mw'].isna() | test_data_24h['pred_24h'].isna())
data_24h_clean = test_data_24h[mask_24h_temporal]

hourly_metrics_24h = data_24h_clean.groupby('target_hour').apply(
    lambda g: pd.Series({
        'mape': mean_absolute_percentage_error(g['target_demand_mw'], g['pred_24h']) * 100,
        'mae': mean_absolute_error(g['target_demand_mw'], g['pred_24h']),
        'rmse': np.sqrt(mean_squared_error(g['target_demand_mw'], g['pred_24h'])),
        'bias_pct': ((g['pred_24h'] - g['target_demand_mw']).mean() / g['target_demand_mw'].mean()) * 100,
        'avg_demand': g['target_demand_mw'].mean(),
        'n_samples': len(g)
    })
).reset_index().rename(columns={'target_hour': 'hour'})
hourly_metrics_24h['model_type'] = '24h'

# Modèle 7j - filtrer les NaN avant le groupby
mask_7d_temporal = ~(test_data_7j['target_demand_mw'].isna() | test_data_7j['pred_7d'].isna())
data_7d_clean = test_data_7j[mask_7d_temporal]

hourly_metrics_7d = data_7d_clean.groupby('target_hour').apply(
    lambda g: pd.Series({
        'mape': mean_absolute_percentage_error(g['target_demand_mw'], g['pred_7d']) * 100,
        'mae': mean_absolute_error(g['target_demand_mw'], g['pred_7d']),
        'rmse': np.sqrt(mean_squared_error(g['target_demand_mw'], g['pred_7d'])),
        'bias_pct': ((g['pred_7d'] - g['target_demand_mw']).mean() / g['target_demand_mw'].mean()) * 100,
        'avg_demand': g['target_demand_mw'].mean(),
        'n_samples': len(g)
    })
).reset_index().rename(columns={'target_hour': 'hour'})
hourly_metrics_7d['model_type'] = '7j'

print(f"   ✅ 24h: {len(hourly_metrics_24h)} heures")
print(f"   ✅ 7j: {len(hourly_metrics_7d)} heures")

# B. Métriques par jour de la semaine
print("\n📅 Métriques par jour de la semaine...")

# Modèle 24h - utiliser les données déjà filtrées
daily_metrics_24h = data_24h_clean.groupby('target_dayofweek').apply(
    lambda g: pd.Series({
        'mape': mean_absolute_percentage_error(g['target_demand_mw'], g['pred_24h']) * 100,
        'mae': mean_absolute_error(g['target_demand_mw'], g['pred_24h']),
        'rmse': np.sqrt(mean_squared_error(g['target_demand_mw'], g['pred_24h'])),
        'avg_demand': g['target_demand_mw'].mean(),
        'n_samples': len(g)
    })
).reset_index().rename(columns={'target_dayofweek': 'day_of_week'})
daily_metrics_24h['model_type'] = '24h'
daily_metrics_24h['day_name'] = daily_metrics_24h['day_of_week'].map({
    0: 'Lundi', 1: 'Mardi', 2: 'Mercredi', 3: 'Jeudi', 4: 'Vendredi', 5: 'Samedi', 6: 'Dimanche'
})

# Modèle 7j - utiliser les données déjà filtrées
daily_metrics_7d = data_7d_clean.groupby('target_dayofweek').apply(
    lambda g: pd.Series({
        'mape': mean_absolute_percentage_error(g['target_demand_mw'], g['pred_7d']) * 100,
        'mae': mean_absolute_error(g['target_demand_mw'], g['pred_7d']),
        'rmse': np.sqrt(mean_squared_error(g['target_demand_mw'], g['pred_7d'])),
        'avg_demand': g['target_demand_mw'].mean(),
        'n_samples': len(g)
    })
).reset_index().rename(columns={'target_dayofweek': 'day_of_week'})
daily_metrics_7d['model_type'] = '7j'
daily_metrics_7d['day_name'] = daily_metrics_7d['day_of_week'].map({
    0: 'Lundi', 1: 'Mardi', 2: 'Mercredi', 3: 'Jeudi', 4: 'Vendredi', 5: 'Samedi', 6: 'Dimanche'
})

print(f"   ✅ 24h: {len(daily_metrics_24h)} jours")
print(f"   ✅ 7j: {len(daily_metrics_7d)} jours")

# C. Métriques Weekday vs Weekend
print("\n🗓️ Métriques Weekday vs Weekend...")

# Modèle 24h - utiliser les données déjà filtrées
weekend_metrics_24h = []
for is_weekend_val in [0, 1]:
    subset = data_24h_clean[data_24h_clean['target_is_weekend'] == is_weekend_val]
    if len(subset) > 0:
        metrics = {
            'day_type': 'Weekend' if is_weekend_val == 1 else 'Weekday',
            'mape': mean_absolute_percentage_error(subset['target_demand_mw'], subset['pred_24h']) * 100,
            'mae': mean_absolute_error(subset['target_demand_mw'], subset['pred_24h']),
            'rmse': np.sqrt(mean_squared_error(subset['target_demand_mw'], subset['pred_24h'])),
            'avg_demand': subset['target_demand_mw'].mean(),
            'n_samples': len(subset),
            'model_type': '24h'
        }
        weekend_metrics_24h.append(metrics)
weekend_df_24h = pd.DataFrame(weekend_metrics_24h)

# Modèle 7j - utiliser les données déjà filtrées
weekend_metrics_7d = []
for is_weekend_val in [0, 1]:
    subset = data_7d_clean[data_7d_clean['target_is_weekend'] == is_weekend_val]
    if len(subset) > 0:
        metrics = {
            'day_type': 'Weekend' if is_weekend_val == 1 else 'Weekday',
            'mape': mean_absolute_percentage_error(subset['target_demand_mw'], subset['pred_7d']) * 100,
            'mae': mean_absolute_error(subset['target_demand_mw'], subset['pred_7d']),
            'rmse': np.sqrt(mean_squared_error(subset['target_demand_mw'], subset['pred_7d'])),
            'avg_demand': subset['target_demand_mw'].mean(),
            'n_samples': len(subset),
            'model_type': '7j'
        }
        weekend_metrics_7d.append(metrics)
weekend_df_7d = pd.DataFrame(weekend_metrics_7d)

print(f"   ✅ 24h: {len(weekend_df_24h)} catégories")
print(f"   ✅ 7j: {len(weekend_df_7d)} catégories")

# Affichage des résultats
print("\n" + "="*80)
print("📊 RÉSULTATS PAR PATTERNS TEMPORELS")
print("="*80)

print("\n🕒 5 heures les plus difficiles (modèle 24h):")
display(hourly_metrics_24h.nlargest(5, 'mape')[['hour', 'mape', 'mae', 'avg_demand', 'n_samples']])

print("\n📅 Performance par jour de la semaine (modèle 24h):")
display(daily_metrics_24h[['day_name', 'mape', 'mae', 'avg_demand', 'n_samples']])

print("\n🗓️ Weekday vs Weekend (modèle 24h):")
display(weekend_df_24h[['day_type', 'mape', 'mae', 'avg_demand', 'n_samples']])

# Visualisations
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Graphique 1: MAPE par heure (24h)
ax1 = axes[0, 0]
ax1.plot(hourly_metrics_24h['hour'], hourly_metrics_24h['mape'], marker='o', linewidth=2, color='blue', label='24h')
ax1.plot(hourly_metrics_7d['hour'], hourly_metrics_7d['mape'], marker='s', linewidth=2, color='green', alpha=0.6, label='7j')
ax1.axhline(y=2.0, color='red', linestyle='--', alpha=0.5, label='Cible 24h')
ax1.set_xlabel('Heure du jour')
ax1.set_ylabel('MAPE (%)')
ax1.set_title('🕒 MAPE par heure du jour', fontweight='bold')
ax1.set_xticks(range(0, 24, 2))
ax1.legend()
ax1.grid(True, alpha=0.3)

# Graphique 2: Demande moyenne par heure avec MAPE
ax2 = axes[0, 1]
ax2_twin = ax2.twinx()
ax2.bar(hourly_metrics_24h['hour'], hourly_metrics_24h['avg_demand'], alpha=0.3, color='gray', label='Demande moyenne')
ax2_twin.plot(hourly_metrics_24h['hour'], hourly_metrics_24h['mape'], marker='o', linewidth=2, color='blue', label='MAPE')
ax2.set_xlabel('Heure du jour')
ax2.set_ylabel('Demande moyenne (MW)', color='gray')
ax2_twin.set_ylabel('MAPE (%)', color='blue')
ax2.set_title('📊 Demande vs Précision par heure (24h)', fontweight='bold')
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')
ax2.grid(True, alpha=0.3, axis='y')

# Graphique 3: MAPE par jour de la semaine
ax3 = axes[1, 0]
bar_width = 0.35
x_pos = np.arange(len(daily_metrics_24h))
ax3.bar(x_pos - bar_width/2, daily_metrics_24h['mape'], bar_width, label='24h', color='blue', alpha=0.7)
ax3.bar(x_pos + bar_width/2, daily_metrics_7d['mape'], bar_width, label='7j', color='green', alpha=0.7)
ax3.set_xlabel('Jour de la semaine')
ax3.set_ylabel('MAPE (%)')
ax3.set_title('📅 MAPE par jour de la semaine', fontweight='bold')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(daily_metrics_24h['day_name'], rotation=45)
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# Graphique 4: Weekday vs Weekend
ax4 = axes[1, 1]
x_pos = np.arange(len(weekend_df_24h))
ax4.bar(x_pos - bar_width/2, weekend_df_24h['mape'], bar_width, label='24h', color='blue', alpha=0.7)
ax4.bar(x_pos + bar_width/2, weekend_df_7d['mape'], bar_width, label='7j', color='green', alpha=0.7)
ax4.set_xlabel('Type de jour')
ax4.set_ylabel('MAPE (%)')
ax4.set_title('🗓️ MAPE Weekday vs Weekend', fontweight='bold')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(weekend_df_24h['day_type'])
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✅ Métriques par patterns temporels calculées")

## 8. Métriques par Horizon de Prévision (Modèle 7j)

In [0]:
# =============================================================================
# 8. MÉTRIQUES PAR HORIZON DE PRÉVISION (MODÈLE 7J)
# =============================================================================
print("\n\n⏱️ CALCUL DES MÉTRIQUES PAR HORIZON DE PRÉVISION (modèle 7j)\n")

# Checkpoints d'horizon pertinents
horizon_checkpoints = [1, 6, 12, 24, 48, 72, 96, 120, 144, 168]
available_horizons = [h for h in horizon_checkpoints if h <= test_data_7j['forecast_horizon_hours'].max()]

print(f"Horizons à évaluer: {available_horizons}")

horizon_metrics = []
for h in available_horizons:
    subset = test_data_7j[test_data_7j['forecast_horizon_hours'] == h]
    if len(subset) > 0:
        # Filtrer les NaN avant calcul des métriques
        mask = ~(subset['target_demand_mw'].isna() | subset['pred_7d'].isna())
        subset_clean = subset[mask]
        
        if len(subset_clean) > 0:
            metrics = {
                'horizon_hours': h,
                'horizon_label': f'H+{h}',
                'mape': mean_absolute_percentage_error(subset_clean['target_demand_mw'], subset_clean['pred_7d']) * 100,
                'mae': mean_absolute_error(subset_clean['target_demand_mw'], subset_clean['pred_7d']),
                'rmse': np.sqrt(mean_squared_error(subset_clean['target_demand_mw'], subset_clean['pred_7d'])),
                'bias_pct': ((subset_clean['pred_7d'] - subset_clean['target_demand_mw']).mean() / subset_clean['target_demand_mw'].mean()) * 100,
                'r2': r2_score(subset_clean['target_demand_mw'], subset_clean['pred_7d']),
                'avg_demand': subset_clean['target_demand_mw'].mean(),
                'n_samples': len(subset_clean)
            }
            horizon_metrics.append(metrics)

horizon_df = pd.DataFrame(horizon_metrics)

print(f"\n✅ Métriques calculées pour {len(horizon_df)} horizons")

# Affichage du tableau
print("\n" + "="*80)
print("⏱️ PERFORMANCE PAR HORIZON DE PRÉVISION")
print("="*80)
display(horizon_df[['horizon_label', 'mape', 'mae', 'rmse', 'r2', 'avg_demand', 'n_samples']])

# Analyse de la dégradation
if len(horizon_df) >= 2:
    h1_mape = horizon_df[horizon_df['horizon_hours'] == min(available_horizons)]['mape'].values[0]
    h_max_mape = horizon_df[horizon_df['horizon_hours'] == max(available_horizons)]['mape'].values[0]
    degradation = h_max_mape - h1_mape
    degradation_pct = (degradation / h1_mape) * 100
    
    print(f"\n📉 ANALYSE DE LA DÉGRADATION:")
    print(f"   H+{min(available_horizons)}: {h1_mape:.2f}% MAPE")
    print(f"   H+{max(available_horizons)}: {h_max_mape:.2f}% MAPE")
    print(f"   Dégradation: +{degradation:.2f} points (+{degradation_pct:.1f}%)")
    
    # Régression linéaire pour tendance
    from scipy.stats import linregress
    slope, intercept, r_value, _, _ = linregress(horizon_df['horizon_hours'], horizon_df['mape'])
    print(f"\n   Tendance linéaire: {'+' if slope > 0 else ''}{slope:.4f}% MAPE par heure")
    print(f"   R²: {r_value**2:.3f}")
    if r_value**2 > 0.7:
        print(f"   🎯 Forte corrélation linéaire (R² > 0.7) - dégradation prévisible")

# Visualisations
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Graphique 1: MAPE par horizon
ax1 = axes[0, 0]
ax1.plot(horizon_df['horizon_hours'], horizon_df['mape'], marker='o', linewidth=3, markersize=10, color='green')
ax1.axhline(y=2.0, color='blue', linestyle='--', alpha=0.5, label='Cible 24h (2%)')
ax1.axhline(y=3.5, color='orange', linestyle='--', alpha=0.5, label='Cible 7j (3.5%)')
ax1.set_xlabel('Horizon (heures)', fontsize=12)
ax1.set_ylabel('MAPE (%)', fontsize=12)
ax1.set_title('⏱️ MAPE par horizon de prévision', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Graphique 2: MAE et RMSE par horizon
ax2 = axes[0, 1]
ax2.plot(horizon_df['horizon_hours'], horizon_df['mae'], marker='o', linewidth=2, label='MAE', color='green')
ax2.plot(horizon_df['horizon_hours'], horizon_df['rmse'], marker='s', linewidth=2, label='RMSE', color='red')
ax2.set_xlabel('Horizon (heures)', fontsize=12)
ax2.set_ylabel('Erreur (MW)', fontsize=12)
ax2.set_title('📊 MAE et RMSE par horizon', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Graphique 3: R² par horizon
ax3 = axes[1, 0]
ax3.plot(horizon_df['horizon_hours'], horizon_df['r2'], marker='o', linewidth=3, markersize=10, color='purple')
ax3.set_xlabel('Horizon (heures)', fontsize=12)
ax3.set_ylabel('R²', fontsize=12)
ax3.set_title('🎯 R² par horizon (qualité d\'ajustement)', fontsize=14, fontweight='bold')
ax3.set_ylim([0, 1])
ax3.grid(True, alpha=0.3)

# Graphique 4: Toutes les métriques normalisées
ax4 = axes[1, 1]
# Normaliser les métriques pour les comparer sur le même graphe
mape_norm = (horizon_df['mape'] - horizon_df['mape'].min()) / (horizon_df['mape'].max() - horizon_df['mape'].min())
mae_norm = (horizon_df['mae'] - horizon_df['mae'].min()) / (horizon_df['mae'].max() - horizon_df['mae'].min())
r2_norm = (horizon_df['r2'] - horizon_df['r2'].min()) / (horizon_df['r2'].max() - horizon_df['r2'].min())

ax4.plot(horizon_df['horizon_hours'], mape_norm, marker='o', linewidth=2, label='MAPE (norm)', color='green')
ax4.plot(horizon_df['horizon_hours'], mae_norm, marker='s', linewidth=2, label='MAE (norm)', color='orange')
ax4.plot(horizon_df['horizon_hours'], 1 - r2_norm, marker='^', linewidth=2, label='1-R² (norm)', color='purple')
ax4.set_xlabel('Horizon (heures)', fontsize=12)
ax4.set_ylabel('Valeur normalisée', fontsize=12)
ax4.set_title('📈 Évolution comparée des métriques (normalisées)', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Analyse par horizon terminée")

## 9. Feature Importance

In [0]:
# =============================================================================
# 9. FEATURE IMPORTANCE
# =============================================================================
print("\n\n🎯 EXTRACTION DE L'IMPORTANCE DES FEATURES\n")

# Extraire l'importance des features pour chaque modèle
feature_importance_data = []

# Modèle 24h
if hasattr(lgb_model_24h, 'feature_importances_'):
    feature_names_24h = expected_features_24h
    importances_24h = lgb_model_24h.feature_importances_
    
    for fname, imp in zip(feature_names_24h, importances_24h):
        feature_importance_data.append({
            'model_type': '24h',
            'feature_name': fname,
            'importance': float(imp)
        })
    
    print(f"✅ Modèle 24h: {len(importances_24h)} features")

# Modèle 7j
if hasattr(lgb_model_7d, 'feature_importances_'):
    feature_names_7d = expected_features_7d
    importances_7d = lgb_model_7d.feature_importances_
    
    for fname, imp in zip(feature_names_7d, importances_7d):
        feature_importance_data.append({
            'model_type': '7j',
            'feature_name': fname,
            'importance': float(imp)
        })
    
    print(f"✅ Modèle 7j: {len(importances_7d)} features")

feature_importance_df = pd.DataFrame(feature_importance_data)

# Top features par modèle
print("\n" + "="*80)
print("🎯 TOP 15 FEATURES LES PLUS IMPORTANTES")
print("="*80)

print("\n🔵 Modèle 24h:")
top_features_24h = feature_importance_df[feature_importance_df['model_type'] == '24h'].nlargest(15, 'importance')
display(top_features_24h)

print("\n🟢 Modèle 7j:")
top_features_7d = feature_importance_df[feature_importance_df['model_type'] == '7j'].nlargest(15, 'importance')
display(top_features_7d)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Modèle 24h
ax1 = axes[0]
top10_24h = feature_importance_df[feature_importance_df['model_type'] == '24h'].nlargest(10, 'importance')
ax1.barh(top10_24h['feature_name'], top10_24h['importance'], color='blue', alpha=0.7)
ax1.set_xlabel('Importance', fontsize=12)
ax1.set_title('🔵 Top 10 Features - Modèle 24h', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Modèle 7j
ax2 = axes[1]
top10_7d = feature_importance_df[feature_importance_df['model_type'] == '7j'].nlargest(10, 'importance')
ax2.barh(top10_7d['feature_name'], top10_7d['importance'], color='green', alpha=0.7)
ax2.set_xlabel('Importance', fontsize=12)
ax2.set_title('🟢 Top 10 Features - Modèle 7j', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n✅ Feature importance extraite")

## 10. Sauvegarde des Résultats dans les Tables de Monitoring

In [0]:
# =============================================================================
# 10. SAUVEGARDE DES RÉSULTATS DANS LES TABLES DE MONITORING
# =============================================================================
print("\n\n💾 SAUVEGARDE DES RÉSULTATS DANS UNITY CATALOG\n")

evaluation_timestamp = pd.Timestamp.now()

# ============================================================================
# Table 1: model_performance_global
# ============================================================================
print("📄 1/7 - model_performance_global...")

global_records = [
    {
        'evaluation_time': evaluation_timestamp,
        'model_type': '24h',
        'model_run_id': run_id_24h,
        'test_period_start': test_data_24h['target_datetime'].min(),
        'test_period_end': test_data_24h['target_datetime'].max(),
        **metrics_24h_global
    },
    {
        'evaluation_time': evaluation_timestamp,
        'model_type': '7j',
        'model_run_id': run_id_7d,
        'test_period_start': test_data_7j['target_datetime'].min(),
        'test_period_end': test_data_7j['target_datetime'].max(),
        **metrics_7d_global
    }
]

df_global = spark.createDataFrame(pd.DataFrame(global_records))
df_global.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(MONITORING_TABLES['global'])
print(f"   ✅ {len(global_records)} lignes sauvées")

# ============================================================================
# Table 2: model_performance_by_zone
# ============================================================================
print("\n📄 2/7 - model_performance_by_zone...")

zone_records = []
for _, row in zone_df_24h.iterrows():
    record = row.to_dict()
    record['evaluation_time'] = evaluation_timestamp
    record['model_run_id'] = run_id_24h
    zone_records.append(record)

for _, row in zone_df_7d.iterrows():
    record = row.to_dict()
    record['evaluation_time'] = evaluation_timestamp
    record['model_run_id'] = run_id_7d
    zone_records.append(record)

df_by_zone = spark.createDataFrame(pd.DataFrame(zone_records))
df_by_zone.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(MONITORING_TABLES['by_zone'])
print(f"   ✅ {len(zone_records)} lignes sauvées")

# ============================================================================
# Table 3: model_performance_by_time
# ============================================================================
print("\n📄 3/7 - model_performance_by_time...")

time_records = []

# Par heure
for _, row in hourly_metrics_24h.iterrows():
    record = row.to_dict()
    record['evaluation_time'] = evaluation_timestamp
    record['model_run_id'] = run_id_24h
    record['time_dimension'] = 'hour'
    time_records.append(record)

for _, row in hourly_metrics_7d.iterrows():
    record = row.to_dict()
    record['evaluation_time'] = evaluation_timestamp
    record['model_run_id'] = run_id_7d
    record['time_dimension'] = 'hour'
    time_records.append(record)

# Par jour de la semaine
for _, row in daily_metrics_24h.iterrows():
    record = row.to_dict()
    record['evaluation_time'] = evaluation_timestamp
    record['model_run_id'] = run_id_24h
    record['time_dimension'] = 'day_of_week'
    time_records.append(record)

for _, row in daily_metrics_7d.iterrows():
    record = row.to_dict()
    record['evaluation_time'] = evaluation_timestamp
    record['model_run_id'] = run_id_7d
    record['time_dimension'] = 'day_of_week'
    time_records.append(record)

# Weekday vs Weekend
for _, row in weekend_df_24h.iterrows():
    record = row.to_dict()
    record['evaluation_time'] = evaluation_timestamp
    record['model_run_id'] = run_id_24h
    record['time_dimension'] = 'day_type'
    time_records.append(record)

for _, row in weekend_df_7d.iterrows():
    record = row.to_dict()
    record['evaluation_time'] = evaluation_timestamp
    record['model_run_id'] = run_id_7d
    record['time_dimension'] = 'day_type'
    time_records.append(record)

df_by_time = spark.createDataFrame(pd.DataFrame(time_records))
df_by_time.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(MONITORING_TABLES['by_time'])
print(f"   ✅ {len(time_records)} lignes sauvées")

# ============================================================================
# Table 4: model_performance_by_horizon
# ============================================================================
print("\n📄 4/7 - model_performance_by_horizon...")

horizon_records = []
for _, row in horizon_df.iterrows():
    record = row.to_dict()
    record['evaluation_time'] = evaluation_timestamp
    record['model_type'] = '7j'
    record['model_run_id'] = run_id_7d
    horizon_records.append(record)

if horizon_records:
    df_by_horizon = spark.createDataFrame(pd.DataFrame(horizon_records))
    df_by_horizon.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(MONITORING_TABLES['by_horizon'])
    print(f"   ✅ {len(horizon_records)} lignes sauvées")
else:
    print(f"   ⚠️  Aucune donnée par horizon")

# ============================================================================
# Table 5: model_error_distribution
# ============================================================================
print("\n📄 5/7 - model_error_distribution...")

error_dist_records = []

# Modèle 24h
errors_24h = test_data_24h['pred_24h'] - test_data_24h['target_demand_mw']
errors_24h_abs = np.abs(errors_24h)
errors_24h_pct = (errors_24h / test_data_24h['target_demand_mw']) * 100

error_dist_records.append({
    'evaluation_time': evaluation_timestamp,
    'model_type': '24h',
    'model_run_id': run_id_24h,
    'error_mean': float(errors_24h.mean()),
    'error_std': float(errors_24h.std()),
    'error_median': float(errors_24h.median()),
    'error_abs_mean': float(errors_24h_abs.mean()),
    'error_abs_median': float(errors_24h_abs.median()),
    'error_pct_mean': float(errors_24h_pct.mean()),
    'error_pct_std': float(errors_24h_pct.std()),
    'error_q01': float(np.percentile(errors_24h_abs, 1)),
    'error_q05': float(np.percentile(errors_24h_abs, 5)),
    'error_q10': float(np.percentile(errors_24h_abs, 10)),
    'error_q25': float(np.percentile(errors_24h_abs, 25)),
    'error_q50': float(np.percentile(errors_24h_abs, 50)),
    'error_q75': float(np.percentile(errors_24h_abs, 75)),
    'error_q90': float(np.percentile(errors_24h_abs, 90)),
    'error_q95': float(np.percentile(errors_24h_abs, 95)),
    'error_q99': float(np.percentile(errors_24h_abs, 99))
})

# Modèle 7j
errors_7d = test_data_7j['pred_7d'] - test_data_7j['target_demand_mw']
errors_7d_abs = np.abs(errors_7d)
errors_7d_pct = (errors_7d / test_data_7j['target_demand_mw']) * 100

error_dist_records.append({
    'evaluation_time': evaluation_timestamp,
    'model_type': '7j',
    'model_run_id': run_id_7d,
    'error_mean': float(errors_7d.mean()),
    'error_std': float(errors_7d.std()),
    'error_median': float(errors_7d.median()),
    'error_abs_mean': float(errors_7d_abs.mean()),
    'error_abs_median': float(errors_7d_abs.median()),
    'error_pct_mean': float(errors_7d_pct.mean()),
    'error_pct_std': float(errors_7d_pct.std()),
    'error_q01': float(np.percentile(errors_7d_abs, 1)),
    'error_q05': float(np.percentile(errors_7d_abs, 5)),
    'error_q10': float(np.percentile(errors_7d_abs, 10)),
    'error_q25': float(np.percentile(errors_7d_abs, 25)),
    'error_q50': float(np.percentile(errors_7d_abs, 50)),
    'error_q75': float(np.percentile(errors_7d_abs, 75)),
    'error_q90': float(np.percentile(errors_7d_abs, 90)),
    'error_q95': float(np.percentile(errors_7d_abs, 95)),
    'error_q99': float(np.percentile(errors_7d_abs, 99))
})

df_error_dist = spark.createDataFrame(pd.DataFrame(error_dist_records))
df_error_dist.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(MONITORING_TABLES['error_dist'])
print(f"   ✅ {len(error_dist_records)} lignes sauvées")

# ============================================================================
# Table 6: model_predictions_sample
# ============================================================================
print("\n📄 6/7 - model_predictions_sample...")

# Échantillon de 1000 prédictions par modèle pour visualisations
sample_24h = test_data_24h.sample(n=min(1000, len(test_data_24h)), random_state=42)
sample_7d = test_data_7j[test_data_7j['forecast_horizon_hours'] == 24].sample(n=min(1000, len(test_data_7j[test_data_7j['forecast_horizon_hours'] == 24])), random_state=42)

pred_sample_records = []

for _, row in sample_24h.iterrows():
    pred_sample_records.append({
        'evaluation_time': evaluation_timestamp,
        'model_type': '24h',
        'model_run_id': run_id_24h,
        'zone': row['zone'],
        'target_datetime': row['target_datetime'],
        'actual_demand_mw': float(row['target_demand_mw']),
        'predicted_demand_mw': float(row['pred_24h']),
        'error_mw': float(row['pred_24h'] - row['target_demand_mw']),
        'error_pct': float(((row['pred_24h'] - row['target_demand_mw']) / row['target_demand_mw']) * 100)
    })

for _, row in sample_7d.iterrows():
    pred_sample_records.append({
        'evaluation_time': evaluation_timestamp,
        'model_type': '7j',
        'model_run_id': run_id_7d,
        'zone': row['zone'],
        'target_datetime': row['target_datetime'],
        'actual_demand_mw': float(row['target_demand_mw']),
        'predicted_demand_mw': float(row['pred_7d']),
        'error_mw': float(row['pred_7d'] - row['target_demand_mw']),
        'error_pct': float(((row['pred_7d'] - row['target_demand_mw']) / row['target_demand_mw']) * 100)
    })

df_pred_sample = spark.createDataFrame(pd.DataFrame(pred_sample_records))
df_pred_sample.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(MONITORING_TABLES['predictions_sample'])
print(f"   ✅ {len(pred_sample_records)} lignes sauvées")

# ============================================================================
# Table 7: model_feature_importance
# ============================================================================
print("\n📄 7/7 - model_feature_importance...")

feature_records = []
for _, row in feature_importance_df.iterrows():
    feature_records.append({
        'evaluation_time': evaluation_timestamp,
        'model_type': row['model_type'],
        'model_run_id': run_id_24h if row['model_type'] == '24h' else run_id_7d,
        'feature_name': row['feature_name'],
        'importance': float(row['importance'])
    })

df_features = spark.createDataFrame(pd.DataFrame(feature_records))
df_features.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(MONITORING_TABLES['feature_importance'])
print(f"   ✅ {len(feature_records)} lignes sauvées")

print("\n" + "="*80)
print("✅ TOUTES LES DONNÉES DE MONITORING ONT ÉTÉ SAUVEGARDÉES")
print("="*80)
print("\nTables créées/mises à jour:")
for name, table in MONITORING_TABLES.items():
    print(f"   ✓ {table}")

## 11. Résumé et Recommandations

In [0]:
# =============================================================================
# 11. RÉSUMÉ ET RECOMMANDATIONS
# =============================================================================
print("\n\n" + "="*80)
print("✅ ÉVALUATION COMPLÈTE TERMINÉE")
print("="*80)

print(f"\n📅 Date d'évaluation: {evaluation_timestamp}")
print(f"\n📌a Données test:")
print(f"   Modèle 24h: {len(test_data_24h):,} échantillons ({test_data_24h['target_datetime'].min()} à {test_data_24h['target_datetime'].max()})")
print(f"   Modèle 7j:  {len(test_data_7j):,} échantillons ({test_data_7j['target_datetime'].min()} à {test_data_7j['target_datetime'].max()})")

print(f"\n🎯 PERFORMANCE GLOBALE:")
print(f"\n   🔵 MODÈLE 24H (run {run_id_24h[:8]}...):")
print(f"      MAPE:       {metrics_24h_global['mape']:.2f}% (cible: < 2.0%)")
print(f"      MAE:        {metrics_24h_global['mae']:,.1f} MW")
print(f"      R²:         {metrics_24h_global['r2']:.3f}")
print(f"      Biais:      {metrics_24h_global['bias_pct']:+.2f}%")
print(f"      Couv. ±2%:  {metrics_24h_global['coverage_2pct']:.1f}%")
if metrics_24h_global['mape'] < 2.0:
    print(f"      ✅ EXCELLENT - Objectif atteint")
elif metrics_24h_global['mape'] < 2.5:
    print(f"      ⚠️  ACCEPTABLE - Légèrement au-dessus de la cible")
else:
    print(f"      ❌ INSUFFISANT - Ré-entraînement recommandé")

print(f"\n   🟢 MODÈLE 7J (run {run_id_7d[:8]}...):")
print(f"      MAPE:       {metrics_7d_global['mape']:.2f}% (cible: < 3.5%)")
print(f"      MAE:        {metrics_7d_global['mae']:,.1f} MW")
print(f"      R²:         {metrics_7d_global['r2']:.3f}")
print(f"      Biais:      {metrics_7d_global['bias_pct']:+.2f}%")
print(f"      Couv. ±5%:  {metrics_7d_global['coverage_5pct']:.1f}%")
if metrics_7d_global['mape'] < 3.5:
    print(f"      ✅ EXCELLENT - Objectif atteint")
elif metrics_7d_global['mape'] < 4.0:
    print(f"      ⚠️  ACCEPTABLE - Légèrement au-dessus de la cible")
else:
    print(f"      ❌ INSUFFISANT - Ré-entraînement recommandé")

print(f"\n💾 TABLES DE MONITORING:")
print(f"   7 tables Unity Catalog créées/mises à jour avec toutes les métriques")
print(f"   Emplacement: {CATALOG}.{SCHEMA}.model_*")

print(f"\n📈 PROCHAINES ÉTAPES:")
print(f"   1. Créer un dashboard Databricks AI/BI pour visualiser ces métriques")
print(f"   2. Planifier ce notebook pour exécution quotidienne/hebdomadaire")
print(f"   3. Configurer des alertes si MAPE dépasse les seuils")

if metrics_24h_global['mape'] >= 2.0 or metrics_7d_global['mape'] >= 3.5:
    print(f"\n⚠️  RECOMMANDATIONS:")
    if metrics_24h_global['mape'] >= 2.0:
        print(f"   - Modèle 24h: Investiguer les zones/heures avec forte erreur")
        print(f"   - Considérer un ré-entraînement avec données plus récentes")
    if metrics_7d_global['mape'] >= 3.5:
        print(f"   - Modèle 7j: Vérifier la dégradation aux horizons lointains")
        print(f"   - Ajouter des features pour améliorer les prévisions à long terme")

print("\n" + "="*80)

In [0]:
# 12. ANALYSE DU MODÈLE 7J PAR HORIZON HORAIRE
print("\n\n🎯 ANALYSE MULTI-HORIZON DU MODÈLE 7J")
print("=" * 80)
print("\nLecture des métriques MLflow par jour de prévision...\n")

client = MlflowClient()
run = client.get_run(run_id_7d)
metrics = run.data.metrics

print(f"📈 Métriques du modèle 7j:")
print(f"   Run ID: {run_id_7d}")

# ============================================================
# PERFORMANCE GLOBALE
# ============================================================
test_mape = metrics.get('test_mape', None)
test_mae = metrics.get('test_mae', None)
test_rmse = metrics.get('test_rmse', None)

if test_mape:
    print(f"\n📊 Performance globale (tous horizons):")
    print(f"   MAPE: {test_mape:.2f}%")
    print(f"   MAE:  {test_mae:.1f} MW")
    print(f"   RMSE: {test_rmse:.1f} MW")
    
    print(f"\n🎯 Évaluation:")
    if test_mape < 3.5:
        print(f"   ✅ EXCELLENTE - cible 7j (3.5%) atteinte")
    elif test_mape < 4.0:
        print(f"   ⚠️ ACCEPTABLE - légèrement au-dessus de la cible")
    else:
        print(f"   ❌ INSUFFISANTE - ré-entraînement recommandé")

# ============================================================
# PERFORMANCE PAR JOUR (J+1 à J+7) AVEC DÉTAIL PAR HEURE
# ============================================================
print("\n⏱️ Performance par jour de prévision (agrégé par tranches de 24h):")
print("=" * 80)

# Extraire toutes les métriques par jour
day_data = []
for day in range(1, 8):  # J+1 à J+7
    day_mape = metrics.get(f'test_day_{day}_mape', None)
    day_mae = metrics.get(f'test_day_{day}_mae', None)
    day_rmse = metrics.get(f'test_day_{day}_rmse', None)
    
    if day_mape is not None:
        day_data.append({
            'day': day,
            'mape': day_mape,
            'mae': day_mae,
            'rmse': day_rmse,
            'horizon_start': (day - 1) * 24 + 1,
            'horizon_end': day * 24
        })

if day_data:
    day_df = pd.DataFrame(day_data)
    
    print(f"\n📋 Tableau détaillé par jour:")
    print(f"{'Jour':<8} {'Horizons':<15} {'MAPE':<10} {'MAE':<12} {'RMSE':<12} {'Statut'}")
    print("-" * 90)
    
    for _, row in day_df.iterrows():
        day = int(row['day'])
        h_start = int(row['horizon_start'])
        h_end = int(row['horizon_end'])
        day_mape = row['mape']
        day_mae = row['mae']
        day_rmse = row['rmse']
        status = '✅' if day_mape < 3.5 else '⚠️' if day_mape < 4.0 else '❌'
        print(f"J+{day:<6} h+{h_start} à h+{h_end:<3}  {day_mape:>6.2f}%    {day_mae:>8.1f} MW  {day_rmse:>8.1f} MW  {status}")
    
    print("-" * 90)
    
    # ============================================================
    # VISUALISATIONS
    # ============================================================
    print("\n📈 Génération des graphiques...")
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    
    # Graphique 1: MAPE par jour
    ax1 = axes[0, 0]
    ax1.bar(day_df['day'], day_df['mape'], color='purple', alpha=0.7, edgecolor='darkviolet', linewidth=2)
    ax1.axhline(y=3.5, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Cible 7j (3.5%)')
    ax1.axhline(y=2.0, color='blue', linestyle='--', linewidth=1.5, alpha=0.5, label='Cible 24h (2.0%)')
    ax1.set_xlabel('Jour de prévision (J+N)', fontsize=12)
    ax1.set_ylabel('MAPE (%)', fontsize=12)
    ax1.set_title('🎯 MAPE par jour de prévision', fontsize=14, fontweight='bold')
    ax1.set_xticks(day_df['day'])
    ax1.set_xticklabels([f'J+{d}' for d in day_df['day']])
    ax1.legend(loc='best', fontsize=10)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Graphique 2: MAE par jour
    ax2 = axes[0, 1]
    ax2.plot(day_df['day'], day_df['mae'], marker='o', linewidth=3, markersize=10, 
             color='green', markerfacecolor='lightgreen', markeredgecolor='darkgreen', markeredgewidth=2)
    ax2.set_xlabel('Jour de prévision (J+N)', fontsize=12)
    ax2.set_ylabel('MAE (MW)', fontsize=12)
    ax2.set_title('📊 MAE par jour de prévision', fontsize=14, fontweight='bold')
    ax2.set_xticks(day_df['day'])
    ax2.set_xticklabels([f'J+{d}' for d in day_df['day']])
    ax2.grid(True, alpha=0.3)
    
    # Graphique 3: RMSE par jour
    ax3 = axes[1, 0]
    ax3.plot(day_df['day'], day_df['rmse'], marker='s', linewidth=3, markersize=10, 
             color='red', markerfacecolor='lightcoral', markeredgecolor='darkred', markeredgewidth=2)
    ax3.set_xlabel('Jour de prévision (J+N)', fontsize=12)
    ax3.set_ylabel('RMSE (MW)', fontsize=12)
    ax3.set_title('📉 RMSE par jour de prévision', fontsize=14, fontweight='bold')
    ax3.set_xticks(day_df['day'])
    ax3.set_xticklabels([f'J+{d}' for d in day_df['day']])
    ax3.grid(True, alpha=0.3)
    
    # Graphique 4: Évolution des 3 métriques ensemble
    ax4 = axes[1, 1]
    ax4_twin1 = ax4.twinx()
    ax4_twin2 = ax4.twinx()
    ax4_twin2.spines['right'].set_position(('outward', 60))
    
    p1 = ax4.plot(day_df['day'], day_df['mape'], marker='o', linewidth=2.5, markersize=8, 
                  color='purple', label='MAPE (%)')
    p2 = ax4_twin1.plot(day_df['day'], day_df['mae'], marker='s', linewidth=2.5, markersize=8, 
                        color='green', label='MAE (MW)')
    p3 = ax4_twin2.plot(day_df['day'], day_df['rmse'], marker='^', linewidth=2.5, markersize=8, 
                        color='red', label='RMSE (MW)')
    
    ax4.set_xlabel('Jour de prévision (J+N)', fontsize=12)
    ax4.set_ylabel('MAPE (%)', fontsize=11, color='purple')
    ax4_twin1.set_ylabel('MAE (MW)', fontsize=11, color='green')
    ax4_twin2.set_ylabel('RMSE (MW)', fontsize=11, color='red')
    ax4.set_title('📊 Évolution des 3 métriques par jour', fontsize=14, fontweight='bold')
    ax4.set_xticks(day_df['day'])
    ax4.set_xticklabels([f'J+{d}' for d in day_df['day']])
    ax4.tick_params(axis='y', labelcolor='purple')
    ax4_twin1.tick_params(axis='y', labelcolor='green')
    ax4_twin2.tick_params(axis='y', labelcolor='red')
    ax4.grid(True, alpha=0.3)
    
    # Légende combinée
    lns = p1 + p2 + p3
    labs = [l.get_label() for l in lns]
    ax4.legend(lns, labs, loc='upper left', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # ============================================================
    # ANALYSE DE LA DÉGRADATION
    # ============================================================
    print("\n🔍 ANALYSE DE LA DÉGRADATION")
    print("=" * 80)
    
    # Dégradation J+1 → J+7
    j1_mape = day_df[day_df['day'] == 1]['mape'].values[0]
    j7_mape = day_df[day_df['day'] == 7]['mape'].values[0]
    
    print(f"\n📉 Dégradation sur la semaine (J+1 → J+7):")
    print(f"   J+1: {j1_mape:.2f}% MAPE (horizons h+1 à h+24)")
    print(f"   J+7: {j7_mape:.2f}% MAPE (horizons h+145 à h+168)")
    print(f"   Δ: +{j7_mape - j1_mape:.2f} points (+{((j7_mape - j1_mape) / j1_mape * 100):.1f}%)")
    
    # Taux de dégradation moyen par jour
    daily_degradation = (j7_mape - j1_mape) / 6
    print(f"   Dégradation moyenne: ~{daily_degradation:.3f}% par jour")
    print(f"   Soit approximativement: ~{daily_degradation / 24:.4f}% par heure")
    
    # Régression linéaire sur l'évolution par jour
    from scipy.stats import linregress
    slope, intercept, r_value, _, _ = linregress(day_df['day'], day_df['mape'])
    
    print(f"\n📈 Tendance linéaire:")
    print(f"   Pente: {'+' if slope > 0 else ''}{slope:.3f}% par jour")
    print(f"   R²: {r_value**2:.3f}")
    
    if slope > 0:
        print(f"   ⚠️ L'erreur AUGMENTE avec l'horizon")
        print(f"   Chaque jour supplémentaire ajoute ~{abs(slope):.2f}% au MAPE")
        if r_value**2 > 0.7:
            print(f"   🎯 Tendance fortement linéaire (R² > 0.7) - dégradation prévisible")
    else:
        print(f"   ✅ L'erreur est stable ou diminue")
    
    # Analyse des seuils par jour
    within_7d_target = day_df[day_df['mape'] < 3.5]
    
    print(f"\n🎯 Atteinte des cibles:")
    print(f"   Jours sous 3.5% (cible acceptable): {len(within_7d_target)} sur {len(day_df)}")
    if len(within_7d_target) > 0:
        print(f"      → Jusqu'à J+{int(within_7d_target['day'].max())} inclus")
        print(f"      → Soit jusqu'à l'horizon h+{int(within_7d_target['horizon_end'].max())}")
    else:
        print(f"      → Aucun jour n'atteint cette cible")
    
    # Statistiques descriptives
    print(f"\n📊 Statistiques descriptives:")
    print(f"   MAPE moyen: {day_df['mape'].mean():.2f}%")
    print(f"   MAPE médian: {day_df['mape'].median():.2f}%")
    print(f"   Écart-type MAPE: {day_df['mape'].std():.2f}%")
    print(f"   Coefficient de variation: {(day_df['mape'].std() / day_df['mape'].mean() * 100):.1f}%")
    
    # Comparaison avec objectifs
    print(f"\n🎯 Comparaison avec les objectifs:")
    avg_mape = day_df['mape'].mean()
    if avg_mape < 3.5:
        print(f"   ✅ Objectif 7j ATTEINT (MAPE moyen: {avg_mape:.2f}% < 3.5%)")
    else:
        gap = avg_mape - 3.5
        print(f"   ❌ Objectif 7j NON ATTEINT (MAPE moyen: {avg_mape:.2f}%)")
        print(f"   Δ à combler: {gap:.2f} points")
        print(f"   Amélioration nécessaire: {(gap / avg_mape * 100):.1f}%")

else:
    print("\n⚠️ Aucune métrique par jour trouvée dans MLflow.")
    print("   Vérifiez que le notebook d'entraînement 07_train_model_7j les a bien enregistrées.")

print("\n💡 Note:")
print("   Cette analyse utilise les métriques agrégées par JOUR (J+1 à J+7).")
print("   Chaque jour représente une tranche de 24 horizons horaires consécutifs.")
print("   Pour une analyse HEURE par HEURE (h+1, h+2, ..., h+168), il faudrait")
print("   ré-entraîner le modèle 7j en enregistrant test_horizon_N_mape dans MLflow.")

print("\n✅ Analyse terminée")
print("=" * 80)

## ✅ Évaluation des modèles

Ce notebook charge les modèles MLflow enregistrés et évalue leurs performances:

1. **Chargement des modèles** depuis MLflow Registry (versions Production ou latest)
2. **Préparation des données de test** (30 derniers jours)
3. **Génération des prédictions** avec les deux modèles (24h et 7j)
4. **Calcul des métriques** (MAE, RMSE, MAPE)
5. **Comparaison des performances** par modèle et par zone
6. **Visualisations** des prédictions vs réalité
7. **Sauvegarde des métriques** dans Unity Catalog pour suivi historique

**Cibles de performance:**
- Modèle 24h: MAPE < 2.0%
- Modèle 7j: MAPE < 3.5%